# Unit 13 exercises: race day

Race day introduces nothing: it asks every earlier unit's question on one known track, `monza-like`, with a car you have not met.
These exercises check one measurement against the line's shape, solve the line again, replay the re-tune's record and catch its floating-point tie, apply the margin rule on every knob at its edge, choose between MPC and pursuit, price a crash, look at the timing policy, and write the packaging checklist.
The Challenge asks whether a faster setting is a finding or a fit.
Each exercise's heading names the concepts it practises.
Every number here is a number about racekit's **carla-like car** and its perturbed family: a model, not CARLA's car.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Results from one exercise are reused in later ones, so run the exercises in order.
Every race goes through the setup's `race()` helper: `agents/unit-13.py` unless a `source` is given, the constants in `constants` changed, **the timing off** (`MPC_BUDGET_MS = None`), on `monza-like`, on `carla_like(noise=True)` (car `k` of Unit 11's perturbed family if `k` is given, and then race seed `k % 5`), **3 laps** unless an exercise says otherwise, cut at 900 s.
A 3-lap MPC race takes about 10 s of computer time, a pursuit race about 5; keep to the races each exercise asks for.
**Never race cars 72 to 95**: they are the unit's test cars, raced once, and racing them again here would spend them.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_07, UNIT_11, UNIT_13 = AGENTS / "unit-07.py", AGENTS / "unit-11.py", AGENTS / "unit-13.py"
workdir = Path(mkdtemp(prefix="roar-unit-13-ex-"))
TRACK = "monza-like"
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
CAP = 900.0  # s: a monza-like race is cut here
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race
MONZA_MU_G = (5.38, 8.96, 9.12, 9.14, 9.70, 7.97)  # m/s^2: Unit 8's six grip probes on monza-like

# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x

NAME_CHARACTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"


def constant_name(line):
    # The UPPER_CASE name of a module-constant line ("NAME = value  # comment"), or None (Unit 12's rule).
    cuts = [i for i in range(len(line) - 2) if line[i:i + 3] == " = "]
    if cuts and line[0] in NAME_CHARACTERS[:26] and all(c in NAME_CHARACTERS for c in line[:cuts[0]]):
        return line[:cuts[0]]
    return None


def module_constants(path):
    # name -> the value's text (after " = ", without a "  # ..." comment), for every module-constant line.
    # A value written over several lines (LINE_CONTROL) is only its first line here: "(".
    constants = {}
    with open(path) as handle:
        for line in handle:
            name = constant_name(line)
            if name is not None:
                text = line[len(name) + 3:-1]
                comment = [i for i in range(len(text) - 2) if text[i:i + 3] == "  #"]
                constants[name] = text[:comment[0]] if comment else text
    return constants


def file_parts(path):
    # (the comment lines above the docstring, the rest of the file without its constant lines and without
    # the line's values): the header may change, the rest must not.
    with open(path) as handle:
        lines = [line for line in handle]
    start = [i for i, line in enumerate(lines) if line[:3] == '"""'][0]
    body, in_line = [], False
    for line in lines[start:]:
        if line[:15] == "LINE_CONTROL = ":
            in_line = True
        elif in_line:
            in_line = line != ")\n"
        elif constant_name(line) is None:
            body.append(line)
    return lines[:start], body


track = load_track(TRACK)
with shim_installed():
    Unit11, name11 = load_submission(UNIT_11)
    Unit13, name13 = load_submission(UNIT_13)
eleven, thirteen = sys.modules[name11], sys.modules[name13]  # the two files' constants and functions
agent13 = Unit13(track.waypoints, None)  # unit-13.py's agent on monza-like (no race): its line, curvature, profile
LINE = np.array(agent13.offsets)  # m: unit-13.py's racing line, as offsets from the centreline
written = []  # every variant file written so far


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def race(source=UNIT_13, seed=0, k=None, laps=3, **constants):
    # One race on monza-like of `source` with `constants` changed and the timing off, on the carla-like car with
    # its noise (car k of Unit 11's perturbed family if k is given: pass seed=k % 5).
    assert k is None or k < 72, "cars 72-95 are the test cars, and k >= 1000 are sealed: not here"
    path = source if source == UNIT_07 else variant(source, **constants, **NO_TIMING)
    return run_race(path, TRACK, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=CAP)


def error_trace(result, offsets=LINE):
    # Unit 9: the time and the cross-track error from the race's own line over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return telemetry["t"][driving], telemetry["offset"][driving] - target


def line_error(result, offsets=LINE):
    return float(np.max(np.abs(error_trace(result, offsets)[1])))


def clean(result, offsets=LINE):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def same_race(a, b):
    # The same race: every telemetry row equal, not only the time.
    return len(a.telemetry) == len(b.telemetry) and all(
        np.sum(a.telemetry[name] != b.telemetry[name]) == 0 for name in a.telemetry.dtype.names)

## Exercise 1 - Section 0's lower bound (`cornering-speed-limit`, `speed-profile`)

Unit 8's probe of section 0 on `monza-like` never slid, so `MU_G[0] = 5.38` is a **lower bound** on that section's grip.
Find out, without a race, whether that can cost any time.
`agent13` (the setup) is `unit-13.py`'s agent built on `monza-like`: `agent13.curvature` is the signed curvature at every point of its racing line (1/m), `agent13.section` the grip section of every point, and `agent13.profile` the speed profile it plans (m/s, Unit 7).
The planner's corner cap in section s is min(`TARGET_SPEED`, √(`MU_FRACTION` · `MU_G[s]` / |κ|)) (Units 7 and 8).
This exercise uses the measured grip; it identifies nothing.

Define:

- `kappa_max`: a dict from each section 0 to 5 to the largest |κ| on the line in that section (1/m);
- `v_max`: a dict from each section to √(`MU_G[s]` / `kappa_max[s]`), the fastest corner speed at the full measured grip (m/s);
- `planner_cap`: a dict from each section to the planner's cap at `kappa_max[s]` (m/s, at most `TARGET_SPEED`);
- `limited`: the sorted list of sections whose `planner_cap` is below `TARGET_SPEED` (where the grip sets the speed);
- `grip_needed`: the grip μ·g (m/s²) below which section 0's cap would drop below `TARGET_SPEED`;
- `high_mu_agent`: the agent of `unit-13.py` with `MU_G[0] = 12.0` (and the other five sections unchanged) built on `monza-like` (`load_submission(variant(UNIT_13, MU_G=...))` inside `shim_installed()`, then the class with `track.waypoints` and `None`), and `profile_same`: `True` when its `profile` equals `agent13.profile`;
- `fig`, `ax`: `agent13.profile` against the waypoint index, with `planner_cap`'s value for each section drawn as a horizontal segment over that section's waypoints, axis labels and a title;
- `why_free`: two sentences: why does section 0's lower bound cost nothing on this line, and when would a lower bound on grip cost time?

In [ ]:
kappa_max, v_max, planner_cap = {}, {}, {}  # TODO
limited = None  # TODO
grip_needed = None  # TODO
high_mu_agent, profile_same = None, None  # TODO
fig, ax = None, None  # TODO
why_free = "TODO"

## Exercise 2 - The line, solved again (`racing-line`, `line-retune`)

A racing line belongs to one track: `unit-11.py` carries `test-circuit`'s and `unit-13.py` carries `monza-like`'s.
Solve `monza-like`'s line with Unit 9's solver (the setup's `solve_line`, about 2 s) and check it against the file.

Define:

- `monza_control`: `solve_line(track)`, and `monza_offsets`: the offset at every waypoint, `spline_basis(len(track)) @ monza_control` (m);
- `room`: `track.half_width - 2.5`, the room the solver allows on each side of the centreline (m), and `inside_width`: `True` when every |offset| is at most its `room` plus 1e-9;
- `pinned_ends`: `True` when every |offset| over the first 30 and the last 30 waypoints is at most 1e-9 (the start and finish stay on the centreline);
- `largest_offset`: the largest |offset| (m);
- `shipped_control`: `thirteen.LINE_CONTROL` as a numpy array, and `same_line`: `np.allclose(monza_control, shipped_control, rtol=0.0, atol=1e-6)` (the line solved here against the 285 values the file ships, each written with `repr`); `max_difference`: the largest |difference| (m);
- `values_per_track`: a dict `{"test-circuit": ..., "monza-like": ...}` with the number of control values in `eleven.LINE_CONTROL` and `thirteen.LINE_CONTROL`;
- `fig`, `ax`: the centreline (`track.x`, `track.y`) and the racing line (each waypoint moved `monza_offsets` along its left normal, `normals(track.x, track.y)`), equal axes, a legend and a title;
- `why_per_track`: two sentences: why can `unit-11.py`'s line not race `monza-like`, and what has to be done again when the race's track changes?

In [ ]:
monza_control, monza_offsets = None, None  # TODO
room, inside_width = None, None  # TODO
pinned_ends = None  # TODO
largest_offset = None  # TODO
shipped_control, same_line, max_difference = None, None, None  # TODO
values_per_track = {}  # TODO
fig, ax = None, None  # TODO
why_per_track = "TODO"

## Exercise 3 - The re-tune's record (`black-box-search`, `repeated-run-measurement`, `determinism-and-packaging`)

`P2` is the probe record of plan `roar-013`: the crash-aware coordinate search on `monza-like`, every setting it evaluated, in order.
Each evaluation raced the nominal car at race seeds 0 and 1 and the training cars 16, 18 and 8 (3 laps); its two **per-replicate sums** are the nominal race at seed r plus the three cars' times.
Each sum is written exactly as the search computed it (with `repr`, Unit 12's trap 1), so your comparisons are the search's.
For a setting that was not clean, the dict gives each failing race: car → (collisions, max line error in m).

**The search** (the lesson's §2, Unit 12's coordinate search): start from `unit-11.py`'s (0.54, 5.0, 4.0); for each knob in the order `MU_FRACTION`, `A_DEC`, `A_ACC`, look at the settings one grid step up and one down (steps 0.02, 0.5, 0.5, each rounded with `round(x, 2)`), take the one with the lower median sum (a setting that is not clean counts as infinitely slow; on a tie, up), and step to it if it **beats** the incumbent: clean, and `noise_aware_better(its sums, the incumbent's sums)`.
After a step, keep stepping the same way while each next setting beats the incumbent; then go on to the next knob.
Repeat the three knobs until a whole pass takes no step.
Every setting your replay needs is in `P2`.

`AT_A_ACC_5` gives four more settings at `A_ACC = 5.0`, raced on the same objective by the record (p3, p3b and p10).

Define:

- `replay(rule)`: the search on `P2` (and `AT_A_ACC_5`), where `rule(candidate_sums, incumbent_sums)` decides "beats" for two clean settings; it returns the list of settings taken, in order, starting with the start;
- `float_steps`: `replay(noise_aware_better)`, and `float_result`: its last setting (the search's literal output);
- `tie_step`: the step of `float_steps` whose two pairs of sums are **one tick apart** in their medians and **one tick** in their spreads (a tuple: the setting before, the setting after);
- `in_ticks`: a function that turns a list of sums into whole ticks (`round(t / DT)`), and `tick_rule(a, b)`: `noise_aware_better(in_ticks(a), in_ticks(b))`;
- `tick_steps`: `replay(tick_rule)`, and `incumbent`: its last setting;
- `edges`: a dict from each knob at an edge to the first value of it that is not clean next to `incumbent` (use `AT_A_ACC_5`), and `no_edge`: the knobs with no edge found;
- `not_clean_reasons`: a dict from each setting of `P2` that is not clean to a list of strings, one per failing race, `"collision"` when it had a collision and `"line error"` when it finished with none but more than 2 m from its line;
- `tie_explained`: three sentences: why does the float comparison say "beats" at `tie_step`, why is it not a gain, and what habit prevents it?

In [ ]:
# The probe record of plan roar-013 (p2): (MU_FRACTION, A_DEC, A_ACC) -> ((the two per-replicate sums in s, exactly
# as computed), {car: (collisions, max line error in m)} for each race that was not clean), in the order evaluated.
P2 = {
    (0.54, 5.0, 4.0): ((1929.9, 1929.8500000000001), {}),
    (0.56, 5.0, 4.0): ((1916.3000000000002, 1916.3000000000002), {}),
    (0.52, 5.0, 4.0): ((1943.9, 1943.75), {}),
    (0.58, 5.0, 4.0): ((1905.7, 1905.65), {}),
    (0.6, 5.0, 4.0): ((1895.5500000000002, 1895.5000000000002), {}),
    (0.62, 5.0, 4.0): ((1886.0, 1885.95), {}),
    (0.64, 5.0, 4.0): ((1876.8500000000001, 1876.8000000000002), {}),
    (0.66, 5.0, 4.0): ((1868.4499999999998, 1868.3999999999999), {}),
    (0.68, 5.0, 4.0): ((1959.6000000000001, 1959.5500000000002), {16: (0, 2.012), 8: (1, 4.510)}),
    (0.66, 5.5, 4.0): ((2130.6000000000004, 2130.6000000000004), {16: (0, 2.794), 18: (0, 2.805), 8: (2, 3.150)}),
    (0.66, 4.5, 4.0): ((1875.7500000000002, 1875.7500000000002), {}),
    (0.66, 5.0, 4.5): ((1862.5500000000002, 1862.6000000000001), {}),
    (0.66, 5.0, 3.5): ((1880.75, 1880.7), {}),
    (0.66, 5.0, 5.0): ((1861.95, 1861.9), {}),
    (0.66, 5.0, 5.5): ((1861.9, 1861.8500000000001), {}),
    (0.66, 5.0, 6.0): ((1862.0500000000002, 1861.95), {}),
    (0.68, 5.0, 5.5): ((2491.6, 2491.45), {16: (2, 6.846), 8: (2, 7.717)}),
    (0.64, 5.0, 5.5): ((1870.3000000000002, 1870.3000000000002), {}),
    (0.66, 5.5, 5.5): ((1856.15, 1856.0500000000002), {16: (0, 2.815), 18: (0, 2.804), 8: (0, 2.780)}),
    (0.66, 4.5, 5.5): ((1869.5000000000002, 1869.5000000000002), {}),
}
# The same objective at A_ACC 5.0, from the record's later probes (p3, p3b, p10), in the same form.
AT_A_ACC_5 = {
    (0.68, 5.0, 5.0): ((2532.9, 2532.8), {16: (2, 4.751), 18: (0, 2.013), 8: (2, 2.951)}),
    (0.64, 5.0, 5.0): ((1870.3000000000002, 1870.3500000000001), {}),
    (0.66, 5.5, 5.0): ((2515.4500000000003, 2515.3500000000004), {16: (2, 9.576), 18: (0, 2.795), 8: (2, 2.815)}),
    (0.66, 4.5, 5.0): ((1869.4, 1869.45), {}),
}
KNOBS = ("MU_FRACTION", "A_DEC", "A_ACC")
STEPS = (0.02, 0.5, 0.5)
START = (0.54, 5.0, 4.0)


def replay(rule):
    return []  # TODO: the settings taken, in order, starting with START


float_steps, float_result = None, None  # TODO
tie_step = None  # TODO
in_ticks, tick_rule = None, None  # TODO
tick_steps, incumbent = None, None  # TODO
edges, no_edge = {}, None  # TODO
not_clean_reasons = {}  # TODO
tie_explained = "TODO"

## Exercise 4 - The margin, on held-out cars (`perturbation-testing`, `held-out-track-validation`)

Unit 12 kept **tracks** apart: training, validation and test.
On race day the track is known and the **car** is not, so the same idea carries from tracks to cars: training cars the objective races, validation cars that judge a setting already found (and are spent once they change a choice), and test cars raced once at the end.

From Exercise 3's `incumbent` and `edges`, apply Unit 11's margin rule (ship one grid step inside the edge) two ways:

- **M**: the margin on Unit 11's knob, `MU_FRACTION`, only;
- **M2**: the margin on **every knob at its edge** (a knob with no edge keeps its value).

Race both on held-out car 41 (`race(seed=41 % 5, k=41, MU_FRACTION=..., A_DEC=..., A_ACC=...)`): two races, about 25 s (M's race runs 686 s of race time).
`HELD_OUT_CLEAN` is the record's count of clean races on the 24 held-out cars k = 24 to 47 (p3, p3b, p3c), and `FRESH` M2's on two more sets.

Define:

- `margin_one_knob`, `margin_every_edge`: the two settings as `(MU_FRACTION, A_DEC, A_ACC)` tuples;
- `car41`: a dict `{"M": result, "M2": result}`, and `car41_clean`, `car41_collisions`: the same with `clean(result)` and the collision count;
- `n_over_n`: the sorted list of the names in `HELD_OUT_CLEAN` that are clean on all 24 held-out cars;
- `which_rule`: `"M"` or `"M2"`, the reading of the rule that is N/N on the held-out cars;
- `sets`: a dict from `"training"`, `"validation"` and `"test"` to the car numbers of each set in this unit, each a `range` (the lesson's §2);
- `spent`: two sentences: why are the cars 24 to 47 a **validation** set now and not a test, and why were cars 48 to 71 not kept as the test either?
- `tracks_to_cars`: two sentences: what plays the role of Unit 12's test **track** here, and what would racing it again (for example to try another setting) do to it?

In [ ]:
# The probe record of plan roar-013 (p3, p3b, p3c): clean races of 24 on the held-out cars k = 24-47, 3 laps.
HELD_OUT_CLEAN = {
    "E (0.66, 5.0, 5.0)": 22, "E' (0.66, 5.0, 5.5)": 22, "M (0.64, 5.0, 5.0)": 23, "M2 (0.64, 4.5, 5.0)": 24,
    "P (0.66, 4.5, 5.0)": 24, "unit-11.py": 24, "unit-12.py": 24, "unit-07.py": 12,
}
# M2 after the choice (p8: cars 48-71, later raced again in p9; p11: cars 72-95, raced once): (clean of 24, median s)
FRESH = {"cars 48-71": (24, 469.65), "cars 72-95": (24, 468.65)}

margin_one_knob, margin_every_edge = None, None  # TODO
car41, car41_clean, car41_collisions = {}, {}, {}  # TODO
n_over_n = None  # TODO
which_rule = "TODO"
sets = {}  # TODO
spent = "TODO"
tracks_to_cars = "TODO"

## Exercise 5 - MPC or pursuit (`ltv-mpc`, `pure-pursuit`)

Unit 10's question at the race-day setting.
Race `unit-13.py` as shipped and the same file with `STEERING = "pursuit"` on held-out car 34 (race seed 4): two races, about 15 s.
`P4` is the record's nominal table (race seeds 0 to 4), and `HELD_OUT` the clean counts on the 24 held-out cars.
Decide by Unit 10's order: **time first** (Unit 1's rule, either way round), **then margin** (the line error and the held-out clean counts).

Define:

- `car34`: a dict `{"mpc": result, "pursuit": result}`; `car34_clean` and `car34_error`: the same with `clean(result)` and `line_error(result)`;
- `faster`: `"mpc"`, `"pursuit"` or `"neither"`, by Unit 1's rule on `P4`'s times;
- `margin_winner`: the steering with the smaller nominal max line error **and** more clean held-out cars (`None` if neither has both);
- `choice`: the steering you ship, by Unit 10's order;
- `cpu_ratio`: MPC's computer time per race over pursuit's (`P4_CPU`);
- `why_mpc`: two sentences: what does MPC buy here that pursuit does not, and why is its cost (`cpu_ratio`) acceptable in the official race?

In [ ]:
# The probe record of plan roar-013 (p4, p3c): the race-day setting, nominal car, race seeds 0-4, 3 laps.
P4 = {"mpc": [468.55, 468.55, 468.55, 468.65, 468.60], "pursuit": [468.55, 468.55, 468.60, 468.65, 468.60]}  # s
P4_MAX_ERROR = {"mpc": 1.009, "pursuit": 1.660}  # m: the largest line error over the five races
P4_CPU = {"mpc": 9.8, "pursuit": 3.6}  # s of computer time per race
HELD_OUT = {"mpc": 24, "pursuit": 13}  # clean of the 24 held-out cars k = 24-47

car34, car34_clean, car34_error = {}, {}, {}  # TODO
faster = "TODO"
margin_winner = None  # TODO
choice = "TODO"
cpu_ratio = None  # TODO
why_mpc = "TODO"

## Exercise 6 - What a crash costs (`respawn-economics`)

Unit 11's **expected race time**: the mean race time over a family of cars, every race counted at its time (a crash costs what the respawn costs), the setting with the shortest wins.
`CAR_TIMES` gives the record's 24 race times on the held-out cars k = 24 to 47 (3 laps, race seed k mod 5) for the edge E (0.66, 5.0, 5.0), M (0.64, 5.0, 5.0), M2 (0.64, 4.5, 5.0) and `unit-07.py`; **a race cut at 900 s counts at 900 s** (the stated crash cost of a race that never finished).
`NOT_CLEAN` names the cars whose race was not clean.
No races.

Define:

- `expected`: a dict from each name to its expected race time (s); `expected_pick`: the name with the shortest;
- `clean_mean_e`: the mean of E's clean races, `crash_mean_e`: the mean of E's races that were not clean, and `saving`: `expected["M2"] - clean_mean_e`, what E saves in a race that does not crash (s);
- `observed_rate_e`: the fraction of E's 24 races that were not clean;
- `break_even_rate`: the crash rate p at which E's expected time, (1 − p) · `clean_mean_e` + p · `crash_mean_e`, equals M2's;
- `fig`, `ax`: the 24 times of each setting against the car number, the not-clean races marked, with a legend, axis labels and a title;
- `checkpoint_b`: three sentences: in Checkpoint B, expected time picked a fraction (0.54) past the family's edge (0.50): by the family's record it would beat the margin rule's 0.48 as long as fewer than 4.5% of races crashed (the family showed 1.7%), yet on race day four of the five sealed cars crashed once at 0.54, and 0.48 won.
  Why does expected time pick the margin setting here instead, and what does the comparison of `observed_rate_e` with `break_even_rate` say?
  What does either rule assume about how often a car crashes on race day?

In [ ]:
# The probe record of plan roar-013 (p3, p3c): 3-lap race times (s) on held-out cars k = 24-47, in order of k.
CAR_TIMES = {
    "E": [737.25, 462.90, 464.95, 465.15, 464.20, 467.20, 465.70, 466.65, 466.95, 463.25, 464.75, 466.40,
          464.10, 463.75, 467.20, 465.75, 465.90, 889.40, 464.80, 464.80, 464.90, 463.10, 463.50, 466.20],
    "M": [466.75, 464.95, 467.05, 467.25, 466.25, 469.20, 467.75, 468.70, 469.05, 465.30, 466.80, 468.40,
          466.25, 465.80, 469.35, 467.75, 467.90, 685.55, 466.85, 466.75, 467.00, 465.10, 465.65, 468.35],
    "M2": [468.60, 466.85, 468.95, 469.15, 468.20, 471.10, 469.75, 470.70, 470.90, 467.25, 468.75, 470.30,
           468.15, 467.75, 471.20, 469.55, 469.85, 467.40, 468.75, 468.75, 468.90, 467.00, 467.55, 470.20],
    "unit-07.py": [900.00, 514.55, 515.35, 515.55, 514.85, 900.00, 515.55, 515.75, 900.00, 900.00, 900.00, 900.00,
                   900.00, 514.90, 516.35, 900.00, 515.50, 900.00, 514.90, 900.00, 515.35, 514.40, 900.00, 900.00],
}
CARS = list(range(24, 48))
NOT_CLEAN = {"E": [24, 41], "M": [41], "M2": [], "unit-07.py": [24, 29, 32, 33, 34, 35, 36, 39, 41, 43, 46, 47]}

expected, expected_pick = {}, None  # TODO
clean_mean_e, crash_mean_e, saving = None, None, None  # TODO
observed_rate_e = None  # TODO
break_even_rate = None  # TODO
fig, ax = None, None  # TODO
checkpoint_b = "TODO"

## Exercise 7 - The timing, on and off (`tick-budget`)

`unit-13.py` ships with the timing on (`MPC_BUDGET_MS = 5.0`) and `MPC_MIN_HORIZON = MPC_HORIZON = 40`.
Race it twice on the nominal car, race seed 0, **1 lap** (`laps=1`, about 7 s in all): once exactly as shipped (`run_race(UNIT_13, TRACK, car=carla_like(noise=True), seed=0, laps=1, max_seconds=CAP)`: the timing on), and once with the timing off (`race(laps=1)`).
Your step times depend on the machine and its load: **print** them and read them; do not compare them with a fixed number.

Then, without a race, build two agents (as in Exercise 1) from `variant(UNIT_13, MPC_BUDGET_MS=0.0)` and `variant(UNIT_13, MPC_BUDGET_MS=0.0, MPC_MIN_HORIZON=10)`, and call each one's `mpc_command(0.0, 15.0)` three times: every call is over a budget of 0 ms.
`P9` is the record's bracket of what Unit 10's halving policy would have raced.

Define:

- `timed`, `untimed`: the two races; `step_ms`: a dict `{"on": (p50, p99), "off": (p50, p99)}` of each race's `step_ms` percentiles (print it);
- `same_time`: `True` when the two race times are equal; `same_telemetry`: `same_race(timed, untimed)`;
- `horizons`: a dict `{40: [...], 10: [...]}` from each `MPC_MIN_HORIZON` to the agent's `horizon` after each of its three calls, and `counted`: a dict from each to its `over_budget` after the three calls;
- `why_same`: two sentences: why are the timing-on and timing-off races the same race with this file, when Unit 12's four timing-on races of `unit-11.py` were four different races?
- `why_counter`: three sentences: from `P9`, was each regime of the halving policy clean, what can no finite set of races cover, and why does the race-day file turn the policy into a counter (and what must then be checked before the race)?

In [ ]:
# The probe record of plan roar-013 (p9): the race-day setting with the horizon fixed at 20, and with
# MPC_BUDGET_MS = 0.0 (every call over budget: the horizon 10 from the third tick), 3 laps, timing off.
# (nominal clean of 5, cars 24-47 clean of 24, cars 48-71 clean of 24, the largest line error in m)
P9 = {"horizon 20": (5, 24, 24, 1.382), "horizon 10 (budget 0.0)": (5, 24, 24, 1.579)}

timed, untimed, step_ms = None, None, {}  # TODO
same_time, same_telemetry = None, None  # TODO
horizons, counted = {}, {}  # TODO
why_same = "TODO"
why_counter = "TODO"

## Exercise 8 - The packaging checklist (`determinism-and-packaging`)

Check the file you would submit as a reviewer would, by reading it.
`module_constants` (the setup) gives every module constant's value as text, without its comment; `file_parts` gives the header comment lines and the rest of the file without its constants and the line's values.

Define:

- `texts`: `module_constants(UNIT_13)`; `pinned_ok`: a dict from each name in `PINNED` to `True` when the file's text for it is exactly the pinned text;
- `round_trips`: a dict from each **float** constant of `PINNED` (`MU_FRACTION`, `A_DEC`, `A_ACC`, `MPC_BUDGET_MS`) to `True` when `float(text) == the value the module holds` (`thirteen.MU_FRACTION` and so on);
- `imports`: the file's import lines (from `file_parts`'s second part: lines starting with `"import "`, or with `"from "` and containing `" import "`), and `allowed_imports`: `True` when each imports only `__future__`, `math`, `time`, `typing`, `numpy` or `roar_py_interface`;
- `header_says`: a dict from each of the six changed constants' names to `True` when it is mentioned in the header;
- `default_report`: `check_submission(UNIT_13)` (its default track, `test-circuit`; cheap: it fails as the agent is built), and `refused_for_the_line`: `True` when the report failed and its text names both 285 and 133;
- `checklist`: a list of at least eight strings, each one thing to check before you submit (the lesson's §6: the file, Python, the values, the timing, the check and where to run it, the versions and the hash, the season's files and wording);
- `why_own_track`: two sentences: why must a race-day file be checked on its own track, and why is the **timed** check run on the race machine, idle?

In [ ]:
# The values the race-day file must carry, as text (plan roar-013, "the reference submission").
PINNED = {"MU_FRACTION": "0.64", "A_DEC": "4.5", "A_ACC": "5.0", "MPC_MIN_HORIZON": "40", "MPC_HORIZON": "40",
          "MPC_BUDGET_MS": "5.0", "STEERING": '"mpc"', "MU_G": "(5.38, 8.96, 9.12, 9.14, 9.7, 7.97)"}

texts, pinned_ok = None, {}  # TODO
round_trips = {}  # TODO
imports, allowed_imports = None, None  # TODO
header_says = {}  # TODO
default_report, refused_for_the_line = None, None  # TODO
checklist = []  # TODO
why_own_track = "TODO"

## Exercise 9 - Challenge: a finding, or a fit? (`held-out-track-validation`, `perturbation-testing`)

P (0.66, 4.5, 5.0) is the margin on `A_DEC` only, the one knob whose next step failed on every low-grip car.
The record (p3b) found it clean on all 24 held-out cars and about 2 s faster than M2 there: median 466.73 s against 468.83 s.
P was raced **after** the held-out cars had already chosen M2, because the table showed a gap worth asking about.

Race P on held-out car 41 (race seed 1) and on the nominal car at race seed 0: two races, about 20 s.
Do **not** race cars 72 to 95 (the `race` helper refuses): they are the test cars, raced once, and nothing you decide here may depend on them.

Define:

- `p_car41`, `p_seed0`: the two results, and `p_clean`: a dict `{"car 41": ..., "seed 0": ...}` of `clean(result)`;
- `gain_seed0`: `unit-13.py`'s seed-0 time, 468.55 s (the milestone's pin), minus P's (s), and `gain_held_out`: 468.83 − 466.73 (s);
- `finding_or_fit`: three sentences: P was chosen by looking at the held-out cars' results; what does that do to their power to judge it, and is 2 s on the cars that suggested it evidence that P is faster on a car you have not met?
- `what_to_race`: two sentences: what would you need to race to tell, how would you fix it **before** racing, and which cars must it never be?

In [ ]:
P_SETTING = {"MU_FRACTION": 0.66, "A_DEC": 4.5, "A_ACC": 5.0}

p_car41, p_seed0, p_clean = None, None, {}  # TODO
gain_seed0, gain_held_out = None, None  # TODO
finding_or_fit = "TODO"
what_to_race = "TODO"